# 10 · Fairness: extra time is not weakness
**Kuizi brain walkthrough, notebook 10 of 10**

Some students have an official **accommodation**: a time multiplier (×1.33 or ×1.5 in the Bac)
and sometimes a screen reader that reads questions aloud. They answer more slowly **for reasons
that have nothing to do with what they know**. A model that reads "slow" as "unsure" would
underestimate them, show them wrong mastery, and route them to easier work they do not need.

Kuizi's rule (T148): time features are divided by the student's time multiplier, assistive use
is never read as hesitation, and every model is checked on these students separately before
promotion (T047).

In the simulator, students with an accommodation know exactly as much as the others; only their
speed differs. So any difference in estimates is unfair by construction.

<div dir="rtl" lang="ar">

## ١٠ · الإنصاف: الوقت الإضافي ليس ضعفًا

بعض التلاميذ لهم <b>تيسيرات</b> رسمية: معامل وقت إضافي (×1.33 أو ×1.5 في الباكالوريا)، وأحيانًا قارئ شاشة يقرأ الأسئلة بصوت عالٍ.
يجيبون أبطأ <b>لأسباب لا علاقة لها بما يعرفون</b>. النموذج الذي يقرأ "البطء" على أنه "عدم يقين" سيقلّل من تقديرهم،
ويعرض عليهم تمكّنًا خاطئًا، ويوجّههم إلى تمارين أسهل لا يحتاجونها.

قاعدة كويزي (T148): نقسم ميزات الزمن على معامل الوقت الخاص بالتلميذ، ولا نقرأ استعمال الأدوات المساعدة على أنه تردّد،
ونفحص كل نموذج على هؤلاء التلاميذ منفصلين قبل اعتماده (T047). في المحاكي، التلاميذ ذوو التيسيرات يعرفون بالضبط مثل غيرهم،
والفرق الوحيد هو السرعة، فأي فرق في التقديرات غير منصف بحكم التصميم.

</div>

In [1]:
# Concept: model settings · Production: T148
EPOCHS, BATCH, LR = 6, 16, 1e-3   # SAINT+ training settings (see notebook 7)

In [2]:
# Concept: same knowledge, different speed · Production: T148, T219
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from kuizi_brain import data as D, models as M, metrics as X
sim = D.simulate_students(n_students=1500, answers_per_student=120, seed=7)
train, test = D.split_by_student(sim)
acc_tr, acc_te = train.time_multiplier.values > 1, test.time_multiplier.values > 1
print(pd.DataFrame({
    "students": [test[acc_te].student.nunique(), test[~acc_te].student.nunique()],
    "true chance of success": [test.p_true[acc_te].mean(), test.p_true[~acc_te].mean()],
    "median seconds per answer": [test.elapsed_s[acc_te].median(), test.elapsed_s[~acc_te].median()],
}, index=["with accommodation", "without"]).round(2))

                    students  true chance of success  \
with accommodation        37                    0.55   
without                  263                    0.54   

                    median seconds per answer  
with accommodation                      88.30  
without                                 66.09  


## A naive "speed = skill" signal

Many products estimate mastery partly from speed ("fast and right means mastered"). Here is the
simplest version: guess success from the answer time alone. Watch what happens to students with
extra time, then divide their times by their multiplier and try again.

<div dir="rtl" lang="ar">

## إشارة ساذجة: "السرعة = المهارة"

كثير من المنتجات تقدّر التمكّن جزئيًا من السرعة ("سريع وصحيح يعني متمكّن"). هذه أبسط نسخة: نخمّن النجاح من زمن الإجابة وحده.
لاحظ ما يحدث للتلاميذ ذوي الوقت الإضافي، ثم نقسم أزمنتهم على معاملهم ونعيد المحاولة.

</div>

In [3]:
# Concept: bias of a speed-based estimate, raw vs normalised · Production: T148, T024
rows = []
for normalised in (False, True):
    def feature(df):
        t = df.elapsed_s / (df.time_multiplier if normalised else 1.0)
        return np.log(t.values)[:, None]
    model = LogisticRegression().fit(feature(train), train.correct)
    p = model.predict_proba(feature(test))[:, 1]
    rows.append({"time feature": "divided by multiplier" if normalised else "raw",
                 "estimate - truth, with accommodation": p[acc_te].mean() - test.p_true[acc_te].mean(),
                 "estimate - truth, without": p[~acc_te].mean() - test.p_true[~acc_te].mean()})
pd.DataFrame(rows).round(3)

,time feature,"estimate - truth, with accommodation","estimate - truth, without"
0,raw,-0.022,0.005
1,divided by multiplier,-0.006,0.002


A negative number means the students are **underestimated**. With raw times, students with extra
time look weaker than they are; dividing by the multiplier removes most of the gap. This is the
whole idea of T148, in one table.

## The same check on SAINT+

Now the real brain, trained once with raw times and once with times divided by the multiplier.
We compare the bias and the AUC for each group of students.

<div dir="rtl" lang="ar">

الرقم السالب يعني أن التلاميذ <b>يُقلَّل من تقديرهم</b>. مع الأزمنة الخام يبدو ذوو الوقت الإضافي أضعف ممّا هم؛ والقسمة على المعامل
تزيل معظم الفرق. هذه فكرة T148 كلّها في جدول واحد.

## نفس الفحص على SAINT+

الآن الدماغ الحقيقي، ندرّبه مرّة بالأزمنة الخام ومرّة بالأزمنة مقسومة على المعامل، ونقارن الانحياز وAUC لكل مجموعة.

</div>

In [4]:
# Concept: SAINT+ with raw vs normalised time features, per group · Production: T148, T047
rows = []
for fair in (False, True):
    torch.manual_seed(0)
    w_train = M.make_windows(train, time_multiplier_fairness=fair)
    w_test = M.make_windows(test, time_multiplier_fairness=fair)
    saint = M.SaintPlus(sim.attrs["n_items"], sim.attrs["n_kc"])
    M.fit(saint, w_train, epochs=EPOCHS, batch_size=BATCH, lr=LR, verbose=False)
    y, p = M.predict(saint, w_test)
    rows.append({"time features": "divided by multiplier" if fair else "raw",
                 "AUC with accommodation": X.auc(y[acc_te], p[acc_te]), "AUC without": X.auc(y[~acc_te], p[~acc_te]),
                 "bias with accommodation": p[acc_te].mean() - test.p_true[acc_te].mean(),
                 "bias without": p[~acc_te].mean() - test.p_true[~acc_te].mean()})
fair_table = pd.DataFrame(rows).round(3)
fair_table

,time features,AUC with accommodation,AUC without,bias with accommodation,bias without
0,raw,0.628,0.63,0.021,0.021
1,divided by multiplier,0.629,0.63,0.021,0.021


In [5]:
# Concept: what the numbers say · Production: T148, T047
raw, fair = fair_table.iloc[0], fair_table.iloc[1]
gap_raw = raw["bias with accommodation"] - raw["bias without"]
gap_fair = fair["bias with accommodation"] - fair["bias without"]
print(f"Gap in bias between the two groups: raw times {gap_raw:+.3f}, normalised times {gap_fair:+.3f}.")
if abs(gap_raw) < 0.01:
    print("Here SAINT+ barely relies on answer time, so the raw model is not visibly unfair on this data.")
    print("The rule still applies: richer time signals in production (hesitation, hint timing) can carry the bias,")
    print("as the speed-only model above shows, and the T047 gate checks every group before any promotion.")
else:
    print("Normalising time by the multiplier is what keeps the two groups on an equal footing.")

Gap in bias between the two groups: raw times +0.000, normalised times +0.000.
Here SAINT+ barely relies on answer time, so the raw model is not visibly unfair on this data.
The rule still applies: richer time signals in production (hesitation, hint timing) can carry the bias,
as the speed-only model above shows, and the T047 gate checks every group before any promotion.


### Remember
- Accommodated students are slower for reasons unrelated to knowledge; a fair brain must not penalise them.
- Divide time features by the official time multiplier; never treat screen-reader time as hesitation (T148).
- Check every model on each group separately; a challenger that hurts a group is not promoted (T047).

**End of the walkthrough.** For the review session (T238), be ready to explain in your own words
how SAINT+ turns a student's answer history into a mastery estimate per concept: notebooks 6, 7 and 9.

<div dir="rtl" lang="ar">

### تذكّر
- التلاميذ ذوو التيسيرات أبطأ لأسباب لا علاقة لها بالمعرفة، والدماغ المنصف لا يعاقبهم.
- اقسم ميزات الزمن على معامل الوقت الرسمي، ولا تعامل وقت قارئ الشاشة على أنه تردّد (T148).
- افحص كل نموذج على كل مجموعة منفصلة؛ المنافس الذي يضرّ بمجموعة لا يُعتمد (T047).

<b>نهاية الدليل.</b> في جلسة المراجعة (T238)، كن مستعدًا لتشرح بكلماتك كيف يحوّل SAINT+ تاريخ إجابات التلميذ إلى تقدير
للتمكّن في كل مفهوم: الدفاتر ٦ و٧ و٩.

</div>